# 1. Introduction

Import libraries:

In [ ]:
import os
import sys
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Check all images are the same dimensions as this affects augmentation results for shifting anatomy centering. In this reasearch, all images are 1008 x 784 pixels.

In [ ]:
input_folder = "path/to/image/folder"

count = 0

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(".jpg"):
        continue
    name, extension = os.path.splitext(filename)
    input_path = os.path.join(input_folder, filename)
    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {input_path}")
    h, w = img.shape[:2]

    if w == 1008 and h == 784:
        continue
        
    print(f"{name}: {w} x {h} pixels")

    count += 1

print(f"Done. {count} files are not 1008 x 784 pixels.")

Define data augmentation functions:
1. increase / reduce magnfication
2. increase / reduce gain
3. shift anatomy centering

In [ ]:
def increase_mag(img, scale=1.2):
    """
    scale > 1.0 magnifies image while keeping image dimensions the same.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape[:2]
    new_w, new_h = int(w * scale), int(h * scale)

    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

    start_x = (new_w - w) // 2
    start_y = (new_h - h) // 2
    magnified = resized[start_y:start_y + h, start_x:start_x + w]
    return magnified

def reduce_mag(img, scale=0.8):
    """
    0 < scale < 1.0 reduces magnification while keeping image dimensions the same.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape[:2]

    # Resize
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Create black canvas of original size
    canvas = np.zeros((h, w), dtype=np.uint8)

    # Compute top-left corner to center the image
    start_x = (w - new_w) // 2
    start_y = (h - new_h) // 2

    # Place resized image onto canvas
    canvas[start_y:start_y + new_h, start_x:start_x + new_w] = resized
    return canvas

def increase_gain(img, factor=1.25, offset=10):
    """
    factor > 1.0 makes the overall image brighter.
    offset > 0 adds a positive brightness offset.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    gain_increased = cv2.convertScaleAbs(img, alpha=factor, beta=offset)
    return gain_increased

def reduce_gain(img, factor=0.8, offset=-10):
    """
    0 < factor < 1.0 makes the overall image dimmer.
    offset < 0 adds a negative brightness offset.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    img = img.astype("float32")

    adjusted = img * factor + offset
    adjusted = np.clip(adjusted, 0, 255).astype("uint8")

    return adjusted

def move_left(img, shift_pixels=100):
    """
    Shift image to left by {shift_pixels} pixels.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape

    # Create black canvas
    canvas = np.zeros((h, w), dtype=np.uint8)

    # Ensure shift is not larger than width
    shift_pixels = min(shift_pixels, w)

    # Move
    canvas[:, :w - shift_pixels] = img[:, shift_pixels:]
    return canvas

def move_right(img, shift_pixels=100):
    """
    Shift image to right by {shift_pixels} pixels.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape

    # Create black canvas
    canvas = np.zeros((h, w), dtype=np.uint8)

    # Ensure shift is not larger than width
    shift_pixels = min(shift_pixels, w)

    # Copy visible region
    # Source: left part of original image
    # Destination: shifted to the right
    canvas[:, shift_pixels:] = img[:, :w - shift_pixels]
    return canvas

def move_up(img, shift_pixels=100):
    """
    Shift image up by {shift_pixels} pixels.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape

    # Create black canvas
    canvas = np.zeros((h, w), dtype=np.uint8)

    # Ensure shift is not larger than width
    shift_pixels = min(shift_pixels, w)

    # Move
    canvas[:h - shift_pixels, :] = img[shift_pixels:, :]
    return canvas

def move_down(img, shift_pixels=100):
    """
    Shift image down by {shift_pixels} pixels.
    """
    if img is None:
        raise ValueError(f"Could not read image.")

    h, w = img.shape

    # Create black canvas
    canvas = np.zeros((h, w), dtype=np.uint8)

    # Ensure shift is not larger than width
    shift_pixels = min(shift_pixels, w)

    # Move
    canvas[shift_pixels:, :] = img[:h - shift_pixels, :]
    return canvas

def blackout_left(img, angle_deg=70):
    """
    Keeps ultrasound fan shape after moving anatomy centering
    by blacking out left side of the screen with rough triangle shape.
    Used after `move_left()`
    """
    if img is None:
        raise ValueError("Input image is None")
    if img.ndim != 2:
        raise ValueError("Input image must be 2D grayscale")
    
    h, w = img.shape[:2]

    # x coordinate based on triangle geometry
    x = int(round(h / np.tan(np.deg2rad(angle_deg))))

    # keep it inside the image
    x = max(0, min(x, w - 1))

    pts = np.array([
        [0, 0],
        [0, h-1],
        [x, 0]
    ], dtype=np.int32)

    blacked_img = img.copy()
    cv2.fillPoly(blacked_img, [pts], 0)
    return blacked_img

def blackout_right(img, angle_deg=70):
    """
    Keeps ultrasound fan shape after moving anatomy centering
    by blacking out right side of the screen with rough triangle shape.
    Used after `move_right()`
    """
    if img is None:
        raise ValueError("Input image is None")
    if img.ndim != 2:
        raise ValueError("Input image must be 2D grayscale")
    
    h, w = img.shape[:2]

    # x coordinate based on triangle geometry
    x = int(round(h / np.tan(np.deg2rad(angle_deg))))

    # keep it inside the image
    x = max(0, min(x, w - 1))

    pts = np.array([
        [w-x, 0],
        [w-1, 0],
        [w-1, h-1]
    ], dtype=np.int32)

    blacked_img = img.copy()
    cv2.fillPoly(blacked_img, [pts], 0)
    return blacked_img

Perform augmentation on one or two images and display results. Use this to inspect if an augmentation produces a realistic ultrasound image.

In [ ]:
input_path1 = "path/to/image0001.jpg"
input_path2 = "path/to/image0002.jpg"

img_spine = cv2.imread(input_path1, cv2.IMREAD_GRAYSCALE)
img_hc = cv2.imread(input_path2, cv2.IMREAD_GRAYSCALE)

img1 = img_spine

img2 = increase_mag(img_spine, scale=1.9)
img2 = increase_gain(img2, factor=1.5, offset=30)

img3 = increase_mag(img_spine, scale=2.5)
img3 = reduce_gain(img3, factor=0.7, offset=-80)
img3 = move_down(img3, shift_pixels=400)

img4 = reduce_mag(img_spine, scale=0.5)
img4 = move_right(img4, shift_pixels=300)
img4 = blackout_right(img4)

img5 = img_hc

img6 = reduce_gain(img_hc, factor=0.6, offset=-60)
img6 = move_left(img6, shift_pixels=300)
img6 = blackout_left(img6)

img7 = reduce_mag(img_hc, scale=0.5)
img7 = move_up(img7, shift_pixels=250)
img7 = increase_gain(img7, factor=1.5, offset=30)

img8 = increase_mag(img_hc, scale=1.5)

In [ ]:
### Display images
# Show original and augmented side by side
plt.figure(figsize=(12, 5.2))
plt.subplot(2, 4, 1)
plt.imshow(img1, cmap="gray")
plt.title("Original - spine")
plt.axis("off")

plt.subplot(2, 4, 2)
plt.imshow(img2, cmap="gray")
# plt.title("2")
plt.axis("off")

plt.subplot(2, 4, 3)
plt.imshow(img3, cmap="gray")
# plt.title("3")
plt.axis("off")

plt.subplot(2, 4, 4)
plt.imshow(img4, cmap="gray")
# plt.title("4")
plt.axis("off")

plt.subplot(2, 4, 5)
plt.imshow(img5, cmap="gray")
plt.title("Original - HC")
plt.axis("off")

plt.subplot(2, 4, 6)
plt.imshow(img6, cmap="gray")
# plt.title("6")
plt.axis("off")

plt.subplot(2, 4, 7)
plt.imshow(img7, cmap="gray")
# plt.title("7")
plt.axis("off")

plt.subplot(2, 4, 8)
plt.imshow(img8, cmap="gray")
# plt.title("8")
plt.axis("off")

plt.tight_layout()

plt.savefig(
    "augmentations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 2. Perform data augmentation in bulk

Three data augmentation steps are run sequentially to apply a range of augmentations to images fulfilling the specified criteria. The steps were designed to produce a balanced mixture of diverse augmentation types and variations in the final dataset. Labels for the augmented images are updated automatically.

As different anatomical views may require slightly different augmentation parameters, each augmentation step should be run separately for each anatomy. Set the `anatomy` and `step` parameters before running each step to keep a record of augmentation results. 

## Step 1

Images that meet the following criteria are processed in step 1:
- good_mag AND good_gain AND good_centering

Step 1 produces reduce_mag variants with 50% chance of increase_gain and reduce_gain for the abovementioned images. 

In [ ]:
anatomy = 6 # integer corresponding to an anatomical view
step = "step1"

# "random.choice(range(-60, -39, 5))" gives a value in steps of 5 from -60 to -40 inclusive
# "random.uniform(-60, -40)" gives a continuous value in range [-60, -40]
INCREASE_MAG_SCALE = random.choice(range(1.7, 1.9, 0.01))
INCREASE_GAIN_FACTOR = random.choice(range(1.4, 1.6, 0.01))
INCREASE_GAIN_OFFSET = random.choice(range(30, 40, 1))
REDUCE_GAIN_FACTOR = random.choice(range(0.6, 0.8, 0.01))
REDUCE_GAIN_OFFSET = random.choice(range(-60, -39, 1)) 

csv_path = "path/to/train/and/validation/annotation/file.csv"
input_folder = "path/to/folder/containing/images/of/spine_sagittal"
output_folder = f"{input_folder}/augmented_{step}"
os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(csv_path)
# Create lookup dictionary
frame_to_label = dict(zip(
    df["frame"],
    zip(df["gnd_ana"], df["gnd_mag"], df["gnd_gain"], df["gnd_centering"], df["gnd_shadow"])
))

count = 0
rows = []

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(".jpg"):
        continue
    name, extension = os.path.splitext(filename)
    label = frame_to_label.get(filename)
    ana, mag, gain, centering, shadow = label

    if ana != anatomy:
        continue

    if mag == 0 and gain == 0 and centering == 0:
        input_path = os.path.join(input_folder, filename)

        img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        img_big = increase_mag(img, scale=INCREASE_MAG_SCALE)
        new_mag = 2

        if count % 2 == 0:
            new_img = increase_gain(img_big, factor=INCREASE_GAIN_FACTOR, offset=INCREASE_GAIN_OFFSET)
            new_gain = 2
        else:
            new_img = reduce_gain(img_big, factor=REDUCE_GAIN_FACTOR, offset=REDUCE_GAIN_OFFSET)
            new_gain = 2

        new_filename = f"{name}_{step}{extension}"
        output_path = os.path.join(output_folder, new_filename)
        cv2.imwrite(output_path, new_img)

        row = {
            "frame": f"{name}_{step}{extension}",
            "gnd_ana": ana,
            "gnd_mag": new_mag,
            "gnd_gain": new_gain,
            "gnd_centering": centering,
            "gnd_shadow": shadow,
        }
        rows.append(row)

        count += 1
        if count % 100 == 0:
            print(f"Processed {count} images")

df_output = pd.DataFrame(rows)
output_file = f"./anatomy{ana}_{step}.csv"
df_output.to_csv(output_file, index=False)

print(f"Done. Processed {count} files.")


## Step 2

Images that meet the following criteria are processed in step 2:
- good_mag AND good_gain AND good_centering

Step 2 produces increase_mag variants with 50% chance of increase_gain and reduce_gain and 20% chance of move_left, move_right, move_up, move_down, and good_centering for the abovementioned images. 

In [ ]:
anatomy = 6 # integer corresponding to an anatomical view
step = "step2"

# "random.choice(range(-60, -39, 5))" gives a value in steps of 5 from -60 to -40 inclusive
# "random.uniform(-60, -40)" gives a continuous value in range [-60, -40]
REDUCE_MAG_SCALE = random.choice(range(0.5, 0.7, 0.01))
MOVE_LEFT = random.choice(range(250, 350, 1))
MOVE_RIGHT = random.choice(range(250, 350, 1))
MOVE_UP = random.choice(range(250, 350, 1))
MOVE_DOWN = random.choice(range(250, 350, 1))
INCREASE_GAIN_FACTOR = random.choice(range(1.4, 1.6, 0.01))
INCREASE_GAIN_OFFSET = random.choice(range(30, 40, 1))
REDUCE_GAIN_FACTOR = random.choice(range(0.6, 0.8, 0.01))
REDUCE_GAIN_OFFSET = random.choice(range(-60, -39, 1)) 

csv_path = "path/to/train/and/validation/annotation/file.csv"
input_folder = "path/to/folder/containing/images/of/spine_sagittal"
output_folder = f"{input_folder}/augmented_{step}"
os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(csv_path)
# Create lookup dictionary
frame_to_label = dict(zip(
    df["frame"],
    zip(df["gnd_ana"], df["gnd_mag"], df["gnd_gain"], df["gnd_centering"], df["gnd_shadow"])
))

count = 0
rows = []

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(".jpg"):
        continue
    name, extension = os.path.splitext(filename)
    label = frame_to_label.get(filename)
    ana, mag, gain, centering, shadow = label

    if ana != anatomy:
        continue

    if mag == 0 and gain == 0 and centering == 0:
        input_path = os.path.join(input_folder, filename)

        img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        img_small = reduce_mag(img, scale=REDUCE_MAG_SCALE)
        new_mag = 1

        if count % 5 == 0:
            img_move = move_left(img_small, shift_pixels=MOVE_LEFT)
            img_move = blackout_left(img_move)
            new_centering = 2
        elif count % 5 == 1:
            img_move = move_right(img_small, shift_pixels=MOVE_RIGHT)
            img_move = blackout_right(img_move)
            new_centering = 1
        elif count % 5 == 2:
            img_move = move_up(img_small, shift_pixels=MOVE_UP)
            new_centering = 4
        elif count % 5 == 3:
            img_move = move_down(img_small, shift_pixels=MOVE_DOWN)
            new_centering = 3
        elif count % 5 == 4: 
            img_move = img_small
            new_centering = 0
        
        if count % 2 == 0:
            new_img = increase_gain(img_move, factor=INCREASE_GAIN_FACTOR, offset=INCREASE_GAIN_OFFSET)
            new_gain = 2
        else:
            new_img = reduce_gain(img_move, factor=REDUCE_GAIN_FACTOR, offset=REDUCE_GAIN_OFFSET)
            new_gain = 1


        new_filename = f"{name}_{step}{extension}"
        output_path = os.path.join(output_folder, new_filename)
        cv2.imwrite(output_path, new_img)

        row = {
            "frame": f"{name}_{step}{extension}",
            "gnd_ana": ana,
            "gnd_mag": new_mag,
            "gnd_gain": new_gain,
            "gnd_centering": new_centering,
            "gnd_shadow": shadow,
        }
        rows.append(row)

        count += 1
        if count % 100 == 0:
            print(f"Processed {count} images")

df_output = pd.DataFrame(rows)
output_file = f"./anatomy{ana}_{step}.csv"
df_output.to_csv(output_file, index=False)

print(f"Done. Processed {count} files.")


## Step 3

Images that meet the following criteria are processed in step 3:
- (NOT reduce_mag) AND good_centering

Step 3 produces off_centered variants (25% chance of move_left, move_right, move_up, and move_down) for the abovementioned images. 

In [ ]:
anatomy = 6 # integer corresponding to an anatomical view
step = "step3"

# "random.choice(range(-60, -39, 5))" gives a value in steps of 5 from -60 to -40 inclusive
# "random.uniform(-60, -40)" gives a continuous value in range [-60, -40]
MOVE_LEFT = random.choice(range(250, 350, 1))
MOVE_RIGHT = random.choice(range(250, 350, 1))
MOVE_UP = random.choice(range(250, 350, 1))
MOVE_DOWN = random.choice(range(250, 350, 1))

csv_path = "path/to/train/and/validation/annotation/file.csv"
input_folder = "path/to/folder/containing/images/of/spine_sagittal"
output_folder = f"{input_folder}/augmented_{step}"
os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(csv_path)
# Create lookup dictionary
frame_to_label = dict(zip(
    df["frame"],
    zip(df["gnd_ana"], df["gnd_mag"], df["gnd_gain"], df["gnd_centering"], df["gnd_shadow"])
))

count = 0
rows = []

for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(".jpg"):
        continue
    name, extension = os.path.splitext(filename)
    label = frame_to_label.get(filename)
    ana, mag, gain, centering, shadow = label

    if ana != anatomy:
        continue

    if mag != 2 and centering == 0:
        input_path = os.path.join(input_folder, filename)

        img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)

        if mag == 0:
            if count % 4 == 0:
                img_move = move_left(img, shift_pixels=MOVE_LEFT)
                new_img = blackout_left(img_move)
                new_centering = 2
            elif count % 4 == 1:
                img_move = move_right(img, shift_pixels=MOVE_RIGHT)
                new_img = blackout_right(img_move)
                new_centering = 1
            elif count % 4 == 2:
                new_img = move_up(img, shift_pixels=MOVE_UP)
                new_centering = 4
            elif count % 4 == 3:
                new_img = move_down(img, shift_pixels=MOVE_DOWN)
                new_centering = 3
        elif mag == 1:
            if count % 4 == 0:
                img_move = move_left(img, shift_pixels=MOVE_LEFT)
                new_img = blackout_left(img_move)
                new_centering = 2
            elif count % 4 == 1:
                img_move = move_right(img, shift_pixels=MOVE_RIGHT)
                new_img = blackout_right(img_move)
                new_centering = 1
            elif count % 4 == 2:
                new_img = move_up(img, shift_pixels=MOVE_UP)
                new_centering = 4
            elif count % 4 == 3:
                new_img = move_down(img, shift_pixels=MOVE_DOWN)
                new_centering = 3

        new_filename = f"{name}_{step}{extension}"
        output_path = os.path.join(output_folder, new_filename)
        cv2.imwrite(output_path, new_img)

        row = {
            "frame": f"{name}_{step}{extension}",
            "gnd_ana": ana,
            "gnd_mag": mag,
            "gnd_gain": gain,
            "gnd_centering": new_centering,
            "gnd_shadow": shadow,
        }
        rows.append(row)

        count += 1
        if count % 100 == 0:
            print(f"Processed {count} images")

df_output = pd.DataFrame(rows)
output_file = f"./anatomy{ana}_{step}.csv"
df_output.to_csv(output_file, index=False)

print(f"Done. Processed {count} files.")
